# Lab: mapping handwritten zip codes

Companion notebook of the lesson *Dimensionality Reduction* — https://insper.github.io/ML/2026.2/classes/dimensionality-reduction/

The lab has two parts. The **fundamental part** (steps 1 to 4) measures how many dimensions the USPS handwritten digits really have, compresses them with PCA and draws them with PCA, t-SNE and UMAP. The **advanced part** (steps 5 to 8) scores the three maps with trustworthiness, varies the knobs of t-SNE and UMAP, places new images on the UMAP map and tests one of the "forbidden statements" about t-SNE/UMAP plots with numbers.

Every step starts with a **Predict**: answer it *before* running the cell. The **Simulations** ask for a small experiment of your own, in the empty cell below them; the solutions are on the lesson page.

**In Colab, install UMAP first** (run once, in a cell of its own): `!pip install umap-learn`. Everything else ships with Colab.

# Fundamental part — steps 1 to 4

## Step 1 — load a sample

**Predict.** Each image is a 16×16 grid of gray pixels. How many columns will the table have? On real envelopes, do you expect the ten digits in equal numbers?

In [ ]:
# In Colab, run this first, in a cell of its own:  !pip install umap-learn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

usps = fetch_openml(data_id=41082, as_frame=True)          # USPS digits, 16x16 pixels
X_all = usps.data.to_numpy()
y_all = usps.target.astype(int).to_numpy() - 1              # OpenML stores digit d as class d + 1
X, X_new, y, y_new = train_test_split(X_all, y_all, train_size=1500, test_size=500,
                                      stratify=y_all, random_state=0)
print(X_all.shape, "->", X.shape, "+", X_new.shape)
print("pixel range:", X.min(), "to", X.max())
print("images per digit:", np.bincount(y))

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for d, ax in enumerate(axes):
    ax.imshow(X[y == d][0].reshape(16, 16), cmap="gray_r")
    ax.set_title(d)
    ax.axis("off")
plt.show()

## Step 2 — how many dimensions does USPS really have?

**Predict.** There are 256 pixels. Will 95% of the variance need close to 256 components, about half of them, or far fewer? What do neighboring pixels have to do with it?

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(X)
cum = np.cumsum(pca.explained_variance_ratio_)
print("first 3 components:", pca.explained_variance_ratio_[:3].round(3))
print("2 components keep:", cum[1].round(3))
for target in [0.90, 0.95, 0.99]:
    print(f"k for {target:.0%}: {np.searchsorted(cum, target) + 1} of 256")

plt.plot(np.arange(1, 257), cum)
plt.axhline(0.95, ls="--", color="gray")
plt.xlabel("number of components k")
plt.ylabel("cumulative explained variance")
plt.show()

**Simulation: standardize the pixels anyway.** [Section 3](https://insper.github.io/ML/2026.2/classes/dimensionality-reduction/#the-scaling-question) warns against standardizing columns that share a unit, such as pixels. Test it: standardize the 256 pixels with `StandardScaler`, fit PCA again and compare how many components 95% of the variance needs. Then compare the standard deviation of the top-left corner pixel with that of the median pixel: what did standardizing do to the corner?

In [ ]:
# your simulation here

## Step 3 — compression and reconstruction error

**Predict.** As \(k\) grows, how fast does the reconstruction error per image fall: steadily, or quickly at first and then slowly? Which numbers from step 2 should it match?

In [ ]:
n = len(X)
print("  k    kept   error/image   discarded eigenvalues")
for k in [1, 2, 5, 10, 20, 50, 100]:
    p = PCA(n_components=k).fit(X)
    X_rec = p.inverse_transform(p.transform(X))
    error = ((X - X_rec) ** 2).sum(axis=1).mean()
    discarded = pca.explained_variance_[k:].sum() * (n - 1) / n
    print(f"{k:3d}  {cum[k - 1]:6.1%}   {error:10.2f}   {discarded:12.2f}")

fig, axes = plt.subplots(1, 6, figsize=(9, 1.8))
for ax, k in zip(axes, [1, 5, 10, 20, 50, 256]):
    p = PCA(n_components=k).fit(X)
    ax.imshow(p.inverse_transform(p.transform(X[:1]))[0].reshape(16, 16), cmap="gray_r")
    ax.set_title(f"k = {k}")
    ax.axis("off")
plt.show()

**Simulation: compress images the PCA never saw.** `transform` lets PCA compress new data with the components learned on the training images. Fit PCA with \(k\) = 5, 20 and 50 on the 1,500 images, then compute the reconstruction error per image on them and on the 500 images of `X_new`, which PCA never saw. Is the error on the new images much larger? What does that say about putting PCA inside a pipeline?

In [ ]:
# your simulation here

## Step 4 — three maps of the same images

**Predict.** Two PCA components keep about a third of the variance. Which of the three maps will separate the ten digits best? Which digits do you expect PCA to keep apart anyway?

In [ ]:
import umap
from sklearn.manifold import TSNE

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit(X)
maps = {"PCA": PCA(n_components=2).fit_transform(X),
        "t-SNE": TSNE(perplexity=30, random_state=0).fit_transform(X),
        "UMAP": reducer.embedding_}
print({name: E.shape for name, E in maps.items()})

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, E) in zip(axes, maps.items()):
    points = ax.scatter(E[:, 0], E[:, 1], c=y, cmap="tab10", s=5)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(points, ax=axes, ticks=range(10))
plt.show()

**To close the fundamental part.** In one sentence each: (1) why do 256 pixels behave like about 30 dimensions? (2) Why is the reconstruction error equal to the variance of the components you dropped? (3) Of the three maps of step 4, which would you use to feed a classifier, and which only to look at the data?

## Your turn (fundamental)

At home:

1. **Choose \(k\) by downstream performance.** [Section 3](https://insper.github.io/ML/2026.2/classes/dimensionality-reduction/#choosing-k) calls it the only valid criterion when PCA feeds a model. Build `make_pipeline(PCA(k), KNeighborsClassifier())` for \(k\) = 2, 5, 10, 20, 50 and 256, fit it on `X, y` and score it on `X_new, y_new`. Which \(k\) wins, and how does it compare with the 32 components of the 95% rule?
2. **Look at the components.** Draw the first eight components (`pca.components_[:8]`, each reshaped to 16×16) with `plt.imshow` and a diverging colormap such as `"RdBu"`. Which strokes does PC1 set against each other? Read pitfall 4 of the [advanced part](https://insper.github.io/ML/2026.2/classes/dimensionality-reduction/#five-pca-pitfalls) before calling any of them a "factor".
3. **On paper, then in code.** Solve [exercise 1](https://insper.github.io/ML/2026.2/classes/dimensionality-reduction/#exercises) by hand, then check your eigenvalues with `np.linalg.eigh` and with `PCA`. Which divisor does each one use?

# Advanced part — steps 5 to 8

## Step 5 — score the maps with trustworthiness

**Predict.** Trustworthiness asks whether each point's neighbors on the map were neighbors in the 256 pixels. Which map will score lowest? As the neighborhood size \(k\) grows, will the nonlinear maps score better or worse?

In [ ]:
from sklearn.manifold import trustworthiness

print("        k=5    k=12   k=30")
for name, E in maps.items():
    print(f"{name:6s}", "  ".join(f"{trustworthiness(X, E, n_neighbors=k):.3f}" for k in [5, 12, 30]))

**Simulation: a map that knows nothing.** [Section 9](https://insper.github.io/ML/2026.2/classes/dimensionality-reduction/#9-a-score-for-a-map-trustworthiness) says that a random map scores about 0.5. Check it on USPS: shuffle the rows of the t-SNE map, so that every image gets the map position of another one, and compute the trustworthiness at \(k = 12\). How far is it from the 0.980 of the real map?

In [ ]:
# your simulation here

## Step 6 — perplexity

**Predict.** Perplexity sets how many neighbors each point considers. From 5 to 100, will the trustworthiness change a lot or a little? And the pictures?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, p in zip(axes, [5, 30, 100]):
    E = maps["t-SNE"] if p == 30 else TSNE(perplexity=p, random_state=0).fit_transform(X)
    print(f"perplexity {p:3d}: trustworthiness {trustworthiness(X, E, n_neighbors=12):.3f}")
    ax.scatter(E[:, 0], E[:, 1], c=y, cmap="tab10", s=5)
    ax.set_title(f"perplexity {p}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## Step 7 — UMAP: `n_neighbors`, `min_dist`, and new images

**Predict.** `min_dist` only sets how tightly points are packed on the map. Which of the two scores, trustworthiness or silhouette, will it move? And will a new image land next to its own digit more often on the UMAP map or in the 256 pixels?

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

print("                          trustworthiness   digits look separated (silhouette)")
for nn, md in [(5, 0.1), (15, 0.1), (50, 0.1), (15, 0.0), (15, 0.5)]:
    E = reducer.embedding_ if (nn, md) == (15, 0.1) else \
        umap.UMAP(n_neighbors=nn, min_dist=md, random_state=0).fit_transform(X)
    print(f"n_neighbors {nn:2d}, min_dist {md}   {trustworthiness(X, E, n_neighbors=12):.3f}"
          f"             {silhouette_score(E, y):.2f}")

E_new = reducer.transform(X_new)                    # 500 images the map has never seen
on_map = NearestNeighbors(n_neighbors=1).fit(reducer.embedding_).kneighbors(E_new, return_distance=False)[:, 0]
in_pixels = NearestNeighbors(n_neighbors=1).fit(X).kneighbors(X_new, return_distance=False)[:, 0]
print(f"new image next to its own digit - on the UMAP map: {(y[on_map] == y_new).mean():.1%}"
      f",  in the 256 pixels: {(y[in_pixels] == y_new).mean():.1%}")

plt.scatter(*reducer.embedding_.T, c="lightgray", s=4)
plt.scatter(*E_new.T, c=y_new, cmap="tab10", s=8)
plt.title("500 new images placed with reducer.transform")
plt.show()

**Simulation: t-SNE has no transform, so start over.** To put the 500 new images on a t-SNE map, the only option is to recompute the map with all 2,000 images. Do it with the same settings (perplexity 30, `random_state=0`). Then measure two things: what share of each old image's 10 map-neighbors survives the refit, and how often a new image lands next to an image of its own digit. Compare with UMAP's `transform` above.

In [ ]:
# your simulation here

## Step 8 — test a forbidden statement: "a bigger group varies more"

**Predict.** A 1 is a single stroke. In the 256-pixel space, will the 1s be the tightest group? And on each of the three maps?

In [ ]:
def spread(Z):                       # root-mean-square distance to the group's center
    return np.sqrt(((Z - Z.mean(axis=0)) ** 2).sum(axis=1).mean())

print("digit   pixels   PCA   t-SNE   UMAP     (each group's spread / average spread)")
rel = {}
for name, E in [("pixels", X)] + list(maps.items()):
    s = np.array([spread(E[y == d]) for d in range(10)])
    rel[name] = s / s.mean()
for d in range(10):
    print(f"{d:5d}   " + "   ".join(f"{rel[name][d]:5.2f}" for name in rel))

**To close the lesson.** In one sentence each: (1) why does a high trustworthiness not prove that the groups on a map exist? (2) Why does `min_dist` change how convincing a UMAP map looks without changing who is a neighbor of whom? (3) Why can UMAP place new images while t-SNE has to start over?

## Your turn (advanced)

At home, with more time:

1. **The null test.** Shuffle each pixel column independently (`rng.permuted(X, axis=0)` with `rng = np.random.default_rng(0)`), run t-SNE on the shuffled images, and plot the real and shuffled maps side by side. Does the shuffled map still show groups? Compare the two trustworthiness scores at \(k = 12\). Here, unlike the 10-column noise of the lesson, the score separates the two cases clearly; why does it work with 256 columns?
2. **PCA before t-SNE.** Reduce to 50 components with PCA, then run t-SNE on those 50 columns. Time both versions with `time.perf_counter()` and compute trustworthiness against the *256-pixel* space (not against the 50 components, which would hide what the first reduction lost). Was the speed-up worth it?
3. **Statement 2.** Compute the ten digit centers in the pixel space and on each map, and the rank correlation between the 45 center-to-center distances (`scipy.stats.spearmanr` on `scipy.spatial.distance.pdist`). Which map preserves the distances between groups best? Does raising the perplexity to 100 help t-SNE?
4. **Seeds.** Run t-SNE with `init="random"` and seeds 0, 1 and 2. Which digit pairs are neighbors in all three maps, and which only in one? Only the first kind is worth reporting.